[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/projective_sphere_flow/projective_sphere_flow.ipynb)

# Diffusion on a Sphere of Rays

This example computes steady heat diffusion on a triangulated sphere whose fields are functions of the rays through its centre. A point of the sphere is modelled as a ray, and each vertex stands for its ray by one chosen representative. Moving a representative along its ray leaves its place on the sphere fixed but changes the rational interpolation across its triangles and the dual cells around its vertices. The heat sources, the surface measure and the conductivity stay those of the fixed unit sphere as one vertex moves, so only the interpolation changes.

A corner function is a quotient of two readings of a point by planes, `(reciprocals & points) / (section & points)`, where each reciprocal plane is the join of the other two corners and `section` is their sum. Its simplex gradient `(reciprocals - barycentrics * section) / (section & points)` pairs to zero with the point itself, so motion along the ray never registers. None of this needs a metric, so the same construction serves the sphere in space and the affine plane of the degenerate projective algebra. The metric enters only with the physical surface, through its quadrature measures and the conductivity, a map from temperature gradients to currents.

In spherical coordinates, steady diffusion reads $\frac{1}{\sin\theta}\,\partial_\theta(\sin\theta\,\partial_\theta T) + \frac{1}{\sin^2\theta}\,\partial_\phi^2 T = -f$, with metric factors that vanish at both poles, and a field interpolated on a flat chord triangle reaches the sphere only through the tangent projection $(\mathbf I - \hat{\mathbf r}\hat{\mathbf r}^{\mathsf T})\nabla$; in homogeneous coordinates the corner functions are the rational barycentric coordinates $\lambda_i(\mathbf x) = w_i\,b_i(\mathbf x)/\sum_j w_j\,b_j(\mathbf x)$ with weights $w_i = 1/s_i$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterable, Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D, PGA2D
from numga.sparse import spdiag
from examples.util import shapes
from examples.util.animation import show_states
from numga.topology import Topology
from examples.fluids.projective_sphere_flow import render, scenarios

ga = VGA3D
ctx = NumpyContext(ga)
mv = ctx.multivector
Scalar = ga.gatype.scalar()                         # Scalar+
Plane = ga.gatype.vector()                          # Vector+
Point = ga.gatype.antivector()                      # Antivector-
Conductivity = ga.gatype((Point, Plane))            # Antivector- <- Vector+

In [ ]:
n_levels = scenarios.n_levels
corner_scales = scenarios.corner_scales
display_grid, display = scenarios.triangle_grid(scenarios.n_display_divisions)  # [3, V_display]
topology, coordinates = shapes.sphere(n_levels)                                 # [V, 3]
V, E, F = topology.n_elements
vertices = mv.vector(coordinates).dual().field()                                # Point-[V]
render.draw_triangulation(topology, vertices);

## Corner functions on rays

The three reciprocal planes read the corner representatives, and dividing by their sum makes the readings add to one everywhere on the projected triangle. These corner functions are called rational barycentric coordinates, and they are constant along each ray. The term ***simplex gradient*** here means the derivative of a corner function across its triangle, a plane that pairs to zero with radial motion.

In [ ]:
triangle = Topology(3, [[0, 1, 2]])
corners = (topology.incidence[2, 0] * vertices).batch()[:, 0]         # [3] Point-
representatives = (corners * corner_scales).field()                   # Point-[V]
tails, heads = triangle.incidence[1, 0] * representatives             # Point-[E], Point-[E]
reciprocals = ((triangle.boundary[2, 1] * (tails & heads)) / (representatives.batch()[0] & representatives.batch()[1] & representatives.batch()[2])).batch()[:, 0]   # [3] Plane+
section = reciprocals.sum(axis=0)                                     # [] Plane+
points = (corners[:, None] * display).sum(axis=0).normalized()        # [...] Point-
denominator = section & points                                        # [...] Scalar+
barycentrics = (reciprocals[:, None] & points) / denominator          # [3, ...] Scalar+
# This quotient derivative kills radial motion, before any metric is supplied.
gradients = (reciprocals[:, None] - barycentrics * section) / denominator   # [3, ...] Plane+
edge_tails, edge_heads = triangle.corners[2, 1]
edge_planes = reciprocals[edge_tails] ^ reciprocals[edge_heads]       # [3] Bivector+
edge_forms = (edge_planes[:, None] & points) / denominator.squared()  # [3, ...] Plane+

Without the denominator, `reciprocals & points` changes along a ray and gives no field on the sphere. The edge reading `edge_forms & displacement` equals the alternating product of one corner function and another's simplex gradient, and registers motion across the projected triangle only.

Changing the corner scales changes this rational interpolation while the sphere itself stays put. Positive scales keep the denominator away from zero inside each triangle, though very unequal scales give strongly skewed interpolations.

## Steady diffusion as a vertex moves along its ray

Heat enters the northern hemisphere and leaves the southern one, at a rate proportional to the height. With uniform conductivity on the unit sphere, that source sustains the height itself as its steady temperature. The material response is integrated on the unit sphere even when the triangles between representatives tilt, so only the interpolation changes.

In [ ]:
n_frames = scenarios.n_frames
log_radius_amplitude = scenarios.log_radius_amplitude
frame_duration_ms = scenarios.frame_duration_ms
quadrature_grid, quadrature_weights = scenarios.triangle_grid(scenarios.n_quadrature_divisions)
# Each small triangle's centroid samples its share of the face, at the mean of its corners' weights.
corner_mean = quadrature_grid.incidence[2, 0].mean(axis=0)        # Scalar+[F_quadrature <- V_quadrature]
quadrature = (corner_mean * mv.scalar(quadrature_weights[..., None]).field()).batch()  # [3, F_quadrature] Scalar+
moving_vertex = int((mv.x & vertices.batch()).argmax())
phases = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
radii = np.exp(log_radius_amplitude * np.sin(phases[:, None]) * (np.arange(V) == moving_vertex))

The adjugate carries the material response back through the same basis used for temperature, so constants produce no current and dissipation is nonnegative.

In [ ]:
def diffusion(topology: Topology, vertices: Point, samples: Point, measures: Scalar,
              source: Scalar, conductivity: Conductivity) -> tuple[Scalar, Plane, Plane]:
    """The vertex potential of steady diffusion under a balanced source on a closed surface, with the face planes whose quotient interpolates it.

    Samples and positive quadrature measures belong to the physical surface; only the interpolation depends on the vertex representatives.
    """
    V, E, F = topology.n_elements
    corners = topology.incidence[2, 0] * vertices                    # [3] Point-[F]
    tails, heads = topology.incidence[1, 0] * vertices               # Point-[E], Point-[E]
    reciprocals = (topology.boundary[2, 1] * (tails & heads)) / (corners[0] & corners[1] & corners[2])  # [3] Plane+[F]
    section = reciprocals.sum(axis=0)                                # Plane+[F]
    denominator = section & samples                                  # [...] Scalar+[F]
    barycentrics = (reciprocals[None, :] & samples[:, None]) / denominator[:, None]  # [..., 3] Scalar+[F]
    # The derivative of the quotient annihilates the radial direction.
    gradients = (reciprocals[None, :] - barycentrics * section) / denominator[:, None]  # [..., 3] Plane+[F]
    interpolation = (spdiag(barycentrics * Scalar) * topology.incidence[2, 0]).sum(axis=1)  # [...] (Scalar+ <- Scalar+)[F <- V]
    gradient = (spdiag(gradients * Scalar) * topology.incidence[2, 0]).sum(axis=1)  # [...] (Plane+ <- Scalar+)[F <- V]

    # The gradient's adjugate pulls the physical material response back to the vertex potentials, integrated on the surface.
    response = measures * conductivity                               # [...] (Point- <- Plane+)[F]
    stiffness = gradient.adjugate()(response(gradient)).sum(axis=0)  # (Antiscalar- <- Scalar+)[V <- V]
    load = interpolation.adjugate()((measures * source).dual()).sum(axis=0)  # Antiscalar-[V]
    mass = interpolation.adjugate()(measures.dual()).sum(axis=0)             # Antiscalar-[V]
    # A closed surface admits a solution only for balanced sources; remove the quadrature mean.
    load = load - mass * (load.dual_inverse().sites.sum() / mass.dual_inverse().sites.sum())
    pin = mv.scalar((np.arange(V) == 0)[..., None]).field() * Scalar.dual()  # (Antiscalar- <- Scalar+)[V]
    potential = (stiffness + pin).solve(load)                        # Scalar+[V]
    potential = potential - (potential & mass).sites.sum() / mass.dual_inverse().sites.sum()
    numerator = (reciprocals * (topology.incidence[2, 0] * potential)).sum(axis=0)  # Plane+[F]
    return potential, numerator, section

The face plane pairs to one with its corner representatives, and its pole `section.dual()` is the dual vertex where the projected dual cells meet.

This cell takes about seven seconds, most of it drawing the animation.

In [ ]:
def spherical_frames(topology: Topology, vertices: Point, radii: Iterable[np.ndarray],
                     quadrature: Scalar, display: np.ndarray
                     ) -> Iterator[tuple[Point, Point, Point, Scalar]]:
    """Yield the representatives, the face poles, the display points and their temperatures, as each frame's corner scales
    choose a new interpolation for the same source and sink."""
    corners = topology.incidence[2, 0] * vertices                          # [3] Point-[F]
    volume = corners[0] & corners[1] & corners[2]                          # Antiscalar-[F]
    n_samples = quadrature.shape[-1]
    quadrature_chords = (corners[:, None] * quadrature).sum(axis=0)        # [n_samples] Point-[F]
    samples = quadrature_chords.normalized()                               # [n_samples] Point-[F]
    # Radial projection expands chord area by the inverse cube of the distance.
    measures = volume.norm() / (2 * n_samples * quadrature_chords.norm() ** 3)   # [n_samples] Scalar+[F]
    source = 2 * (mv.z & samples)                                                # [n_samples] Scalar+[F]
    conductivity = Plane.dual()                                       # [] Point- <- Plane+
    display_points = (corners[:, None] * display).sum(axis=0).normalized() # [...] Point-[F]
    for scales in radii:
        representatives = vertices * mv.scalar(scales[..., None]).field()  # Point-[V]
        _, numerator, section = diffusion(topology, representatives, samples, measures, source, conductivity)
        values = (numerator & display_points) / (section & display_points)       # [...] Scalar+[F]
        yield representatives, section.dual().normalized(), display_points, values

frames = show_states(spherical_frames(topology, vertices, radii, quadrature, display), render.animate_sphere,
                     topology, display_grid, name="projective_sphere_weights", duration_ms=frame_duration_ms)

The left panel shows the blue representatives and the orange polar dual projected onto the sphere; the right panel shows the computed temperature on the fixed sphere. The animation varies only the representation, and the temperature's small departures from the height follow the interpolation while the physical problem stays the same.

The inverse corner radii are the weights of a weighted dual triangulation: changing them moves the dual cells while the spherical vertices and great-circle edges stay fixed, and here the same weights also choose the rational interpolation. The material response comes from those rational fields integrated over the sphere rather than from ratios of dual to primal edge lengths. [Weighted triangulations](https://www.geometry.caltech.edu/pubs/dGMMD14.pdf) and [Hodge-optimized triangulations](https://pages.saclay.inria.fr/mathieu.desbrun/pubs/MMdGD11.pdf) use related freedom to improve primal and dual operators.

## The same quotient in x, y, w

In the degenerate algebra the physical surface is the finite affine slice. Scaling all coordinates of a point by a positive number leaves the point fixed after normalization, while a direction pairs to zero with `w`. Changing only the `w` coordinate while holding `x` and `y` fixed moves the finite point, so it gives a different point rather than another representative. The reciprocal planes and their simplex gradients need only joins and scalar division, and no inverse of the degenerate metric occurs.

In [ ]:
planar_ga = PGA2D
planar_mv = NumpyContext(planar_ga).multivector
PlanarPlane = planar_ga.gatype.vector()             # Vector+
corner_values = scenarios.corner_values
planar_corners = scenarios.planar_corners                             # [3] Point-

In [ ]:
planar_triangle = Topology(3, [[0, 1, 2]])
planar_points = (planar_corners[:, None] * display).sum(axis=0)       # [...] Point-
planar_representatives = (planar_corners * corner_scales).field()     # Point-[V]
tails, heads = planar_triangle.incidence[1, 0] * planar_representatives   # Point-[E], Point-[E]
reciprocals = ((planar_triangle.boundary[2, 1] * (tails & heads)) / (planar_representatives.batch()[0] & planar_representatives.batch()[1] & planar_representatives.batch()[2])).batch()[:, 0]  # [3] Plane+
section = reciprocals.sum(axis=0)                                     # [] Plane+
planar_barycentrics = (reciprocals[:, None] & planar_points) / (section & planar_points)  # [3, ...] Scalar+
planar_gradients = (reciprocals[:, None] - planar_barycentrics * section) / (section & planar_points)  # [3, ...] Plane+
potential = (planar_barycentrics * corner_values[:, None]).sum(axis=0)    # [...] Scalar+
gradient = (planar_gradients * corner_values[:, None]).sum(axis=0)    # [...] Plane+
# The material reads spatial directions, leaving w out of distance without inverting it.
directions = stack((planar_mv.x.dual(), planar_mv.y.dual()))          # [2] Point-
conductivity = (directions * (PlanarPlane & directions)).sum(axis=0)  # [] Point- <- Plane+
current = -conductivity(gradient)                                     # [...] Point-
render.draw_planar(planar_points, display_grid, potential, current);

Colour shows the interpolated potential and the white streaks the current in the physical plane. The material responds to spatial motion only, and the homogeneous coordinate never enters as an extra distance.

Scale enters in two ways. Keeping `section = reciprocals.sum(axis=0)` lets the corner scales choose different rational barycentric coordinates, as on the sphere. Holding a physical chart fixed and compensating the reciprocal planes for the corner weights instead makes the rescaling purely a change of representative, and leaves every corner function unchanged.

Neither construction requires nearly equal weights. A zero denominator marks a chart boundary, where points at infinity remain valid projective points but are no finite locations of this planar diffusion problem.

In [ ]:
# The physical affine chart stays fixed when only representatives change.
chart = planar_mv.w                                                   # [] Plane+
chart_reciprocals = reciprocals * (chart & planar_representatives.batch())   # [3] Plane+
chart_barycentrics = (chart_reciprocals[:, None] & planar_points) / (chart & planar_points)  # [3, ...] Scalar+
# These are the ordinary affine barycentric coordinates, whatever the corner scales.
planar_field = planar_corners.field()                                 # Point-[V]
affine_tails, affine_heads = planar_triangle.incidence[1, 0] * planar_field  # Point-[E], Point-[E]
affine_reciprocals = ((planar_triangle.boundary[2, 1] * (affine_tails & affine_heads)) / (planar_corners[0] & planar_corners[1] & planar_corners[2])).batch()[:, 0]  # [3] Plane+
affine_barycentrics = affine_reciprocals[:, None] & planar_points     # [3, ...] Scalar+

In [ ]:
# checks
# The corner functions partition unity, and their simplex gradients see no motion along the ray.
assert ((barycentrics.sum(axis=0) - 1).abs() < 1e-12).all()
assert ((gradients & points).abs() < 1e-12).all()
alternating = barycentrics[edge_tails] * gradients[edge_heads] - barycentrics[edge_heads] * gradients[edge_tails]
assert ((edge_forms - alternating).norm() < 1e-12).all()
# Under every weighting the temperature stays near the height that the source sustains.
assert all(((sphere_values - (mv.z & sphere_points)).abs() < 0.15).all() for _, _, sphere_points, sphere_values in frames)
# The planar current is a direction, and the fixed chart recovers the affine barycentric coordinates.
assert ((planar_mv.w & current).abs() < 1e-12).all()
assert ((chart_barycentrics - affine_barycentrics).abs() < 1e-12).all()